# Evaluate historical and new diagnosis labels

Configure the input paths in the setup cell, then run the cells in order. See the project README for input schemas and model setup.


In [ ]:
from pathlib import Path
import sys

# Locate the project when Jupyter starts in a notebook subfolder.
project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "smmnlp" / "__init__.py").is_file()),
    None,
)
if project_root is None:
    raise RuntimeError("Start Jupyter inside SMMnlp_v2 or set its working directory there.")
sys.path.insert(0, str(project_root))


In [ ]:
import logging

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay

from smmnlp.evaluation import calculate_binary_metrics, print_metrics
from smmnlp.paths import ProjectPaths
from smmnlp.parsing import parse_history_for_evaluation
from smmnlp.preprocessing import add_parsed_labels

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
paths = ProjectPaths.from_environment()
# Override paths here, or set SMMNLP_DATA_ROOT and SMMNLP_MODEL_PATH.

## Phase 2 results: hx vs. new Training yes SMM n=199 Testing  yes SMM n= 85


## Training & Testing ids


In [ ]:
readfiletrue = paths.data("Update/True/SMMNLP_training_true_phase1.xlsx")
true_training = pd.read_excel(readfiletrue)

In [ ]:
readfiletrue = paths.data("Update/True/SMMNLP_testing_true_phase1.xlsx")
true_testing = pd.read_excel(readfiletrue)

In [ ]:
train_ids = true_training["PatientSSN"].unique()
test_ids = true_testing["PatientSSN"].unique()

In [ ]:
len(train_ids)

In [ ]:
len(test_ids)

## Read in true labels


In [ ]:
truedata = paths.data("Update/True/SMMNLP_testing_true_phase2.xlsx")
true_all = pd.read_excel(truedata)[["PatientSSN", "NoteDate", "NoteText", "HxNew SMM Dx"]]

In [ ]:
true_all.head(5)

In [ ]:
len(true_all)

## Read in newly generated results


In [ ]:
readfileresults = paths.data("Update/ExtraLLMRun/phase2_zeroshot_hxnew_updtesting.xlsx")
results = pd.read_excel(readfileresults)[["PatientSSN", "NoteDate", "NoteText", "HxNew_LLM_Binary"]]

In [ ]:
results.head()

In [ ]:
len(results)

## Combine true and result data for performance


In [ ]:
final = true_all.merge(
    results[["PatientSSN", "NoteDate", "HxNew_LLM_Binary"]],
    on=["PatientSSN", "NoteDate"],
    how="inner",
)

In [ ]:
len(final)

In [ ]:
final.head(5)

## POST PROCESSING


In [ ]:
llm_columns = ["HxNew_LLM_Binary"]
final = add_parsed_labels(final, llm_columns, parse_history_for_evaluation)

In [ ]:
final.head(5)

## Note-level BINARY CLASSIFICATION PERFORMANCE


In [ ]:
conditions = [
    (final["HxNew SMM Dx"] == "new") & (final["post_HxNew_LLM_Binary"] == "new"),
    (final["HxNew SMM Dx"] == "hx") & (final["post_HxNew_LLM_Binary"] == "hx"),
    (final["HxNew SMM Dx"] == "hx") & (final["post_HxNew_LLM_Binary"] == "new"),
    (final["HxNew SMM Dx"] == "new") & (final["post_HxNew_LLM_Binary"] == "hx"),
]
choices = ["True New", "True Hx", "False New", "False Hx"]
final["Outcome2_HxNew"] = np.select(conditions, choices, default="Unknown")

In [ ]:
final.head(5)

In [ ]:
freq_table = final["HxNew SMM Dx"].value_counts()
print(freq_table)

In [ ]:
final_error = final[final["Outcome2_HxNew"].isin(["False New", "False Hx"])]

In [ ]:
final_error["NoteText"] = (
    final_error["NoteText"]
    .str.replace("_x000D_", "", regex=False)
    .str.replace("\r", "\n", regex=False)
)

In [ ]:
len(final_error)

In [ ]:
erroranalysis = paths.data("Update/Results/phase2errors_testing_fewshot.xlsx")
final_error.to_excel(erroranalysis, engine="xlsxwriter", sheet_name="Sheet1", index=False)

In [ ]:
final["y_true"] = final["HxNew SMM Dx"].str.lower().map({"new": 1, "hx": 0})
final["y_pred"] = final["post_HxNew_LLM_Binary"].str.lower().map({"new": 1, "hx": 0})

In [ ]:
y_true = final["y_true"]
y_pred = final["y_pred"]
metric_result = calculate_binary_metrics(y_true, y_pred)

In [ ]:
cm_display = ConfusionMatrixDisplay(
    confusion_matrix=metric_result.cm, display_labels=["Hx SMM Dx", "New SMM Dx"]
)
cm_display.plot()
plt.show()

In [ ]:
print_metrics(metric_result)